In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_33.pickle

In [ ]:
%%cudf.pandas.profile
### cell 33 ###

def grab_subset_of_data_45(original_df, question_of_interest):
    # select only the columns containing the question, strip off the question prefix, and drop all‐NA rows
    subset = (
        original_df
        .filter(like=question_of_interest, axis=1)
        .rename(columns={
            col: col.split('-')[-1].lstrip()
            for col in original_df.columns
            if question_of_interest in col
        })
        .dropna(how='all')
    )
    return subset

# grab and prepare the ML‐frameworks subset on the GPU
ml_frameworks_df_2022_cell45 = grab_subset_of_data_45(
    responses_df_2022_cell10,
    question_of_interest_cell44
)

# define groups of related frameworks and build each combined column in one pass
framework_groups = {
    'TensorFlow/Keras': ['TensorFlow ', 'Keras '],
    'PyTorch/PyTorch Lightning/Fast.ai': ['PyTorch ', 'PyTorch Lightning ', 'Fast.ai '],
    'Xgboost/LightGBM/CatBoost': ['Xgboost ', 'LightGBM ', 'CatBoost ']
}

for new_col, cols in framework_groups.items():
    mask = ml_frameworks_df_2022_cell45[cols].notna().any(axis=1)
    # map True→framework name, False→null in one GPU operation
    ml_frameworks_df_2022_cell45[new_col] = mask.map({True: new_col, False: None})

# drop all the original one‐off columns in a single call
to_drop = [c for cols in framework_groups.values() for c in cols]
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(columns=to_drop)

# keep only the Scikit—learn column plus the new combined ones
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45[
    ['Scikit—learn '] + list(framework_groups.keys())
]

ml_frameworks_df_2022_cell45.info()